# PyTorch & TCN Concepts — Interview Prep

This notebook is purely educational. It does not train a model — it explains
the *why* behind every design decision in `crime_tcn.ipynb` using minimal
self-contained examples. Read this before an interview.

**Sections:**
1. PyTorch Fundamentals — tensors, autograd, `nn.Module`, training loop
2. Why TCN over LSTM — mechanism-level comparison
3. Causal & Dilated Convolutions — visualised
4. Residual Connections — mathematical derivation
5. Hyperparameter Tuning Guide
6. Loss Functions — MSE vs MAE vs Huber
7. Full Pipeline Flowchart (matplotlib)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import torch
import torch.nn as nn

torch.manual_seed(42)
print('PyTorch version:', torch.__version__)

---
## Section 1 — PyTorch Fundamentals

### 1a. Tensors

A `torch.Tensor` is PyTorch's version of a NumPy array. The key difference:
tensors can track gradients. Every arithmetic operation on a tensor that has
`requires_grad=True` is recorded in a **computation graph**.

In [ ]:
# Creating tensors
a = torch.tensor([1.0, 2.0, 3.0])          # from Python list
b = torch.from_numpy(np.array([4.0, 5.0, 6.0]))  # from NumPy
c = torch.zeros(3, 4)                       # shape (3, 4), all zeros
d = torch.randn(2, 3)                       # shape (2, 3), standard normal

print('a:', a)
print('b dtype:', b.dtype)   # float64 from NumPy — often need to cast to float32
print('c shape:', c.shape)
print('d:\n', d)

# Most PyTorch layers require float32 (not float64).
b_f32 = b.float()  # .float() is shorthand for .to(torch.float32)
print('b float32:', b_f32.dtype)

### 1b. Autograd — Automatic Differentiation

When you call `loss.backward()`, PyTorch walks the computation graph backwards
and computes `d(loss)/d(param)` for every parameter using the chain rule.
This is called **reverse-mode automatic differentiation**.

You never write gradient formulas by hand — PyTorch derives them automatically.

In [ ]:
# Simple example: y = x^2, dy/dx = 2x
# We set requires_grad=True so PyTorch records operations on x.
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2          # PyTorch records: y depends on x via squaring
y.backward()        # compute dy/dx — walks the computation graph backwards

print(f'x = {x.item()}')
print(f'y = x^2 = {y.item()}')
print(f'dy/dx (auto) = {x.grad.item()}')   # Should be 2*3 = 6
print(f'dy/dx (manual) = {2 * x.item()}')  # Confirm

In [ ]:
# Neural network analogy: weight w, prediction = w * input, loss = (pred - target)^2
w      = torch.tensor(0.5, requires_grad=True)  # a learnable weight
input_ = torch.tensor(4.0)                       # fixed input
target = torch.tensor(2.0)                       # fixed target

pred = w * input_         # forward pass: prediction
loss = (pred - target)**2 # loss = (2.0 - 2.0)^2 ... or not

loss.backward()  # compute d(loss)/d(w)

print(f'pred = {pred.item():.3f}')
print(f'loss = {loss.item():.3f}')
print(f'd(loss)/d(w) = {w.grad.item():.3f}')
# Manual: d/dw [(w*4 - 2)^2] = 2*(w*4-2)*4 = 2*(2-2)*4 = 0 if w=0.5, else non-zero

# This gradient is exactly what optimizer.step() uses to update w:
lr = 0.01
with torch.no_grad():   # no_grad: don't record this operation in the graph
    w -= lr * w.grad
print(f'w after one update: {w.item():.4f}')

### 1c. `nn.Module` — How PyTorch Models Work

Every model and every layer inherits from `nn.Module`. The two methods you
**must** implement:
- `__init__`: define layers as `self.something = nn.Linear(...)` — PyTorch
  auto-registers any `nn.Module` attribute as a child module (so its parameters
  are found by `model.parameters()`).
- `forward(x)`: defines the computation. You never call `forward()` directly;
  call `model(x)` which invokes `forward()` plus registered hooks.

In [ ]:
class TinyNet(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()   # Required: initialise nn.Module internals
        # Assigning nn.Linear to self.fc registers it as a sub-module.
        # nn.Linear(in, out) applies the transformation: y = W*x + b
        # W has shape (out, in), b has shape (out,). Both are learnable.
        self.fc1 = nn.Linear(in_features, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, out_features)

    def forward(self, x):
        # You define the exact computation here.
        # PyTorch traces this during the forward pass to build the graph.
        x = self.fc1(x)   # (batch, in_features) -> (batch, 32)
        x = self.relu(x)  # apply element-wise max(0, x)
        x = self.fc2(x)   # (batch, 32) -> (batch, out_features)
        return x

net = TinyNet(10, 1)

# Parameters: all learnable tensors across all sub-modules.
n_params = sum(p.numel() for p in net.parameters())
print(f'TinyNet parameters: {n_params}')
# fc1: 10*32 weights + 32 biases = 352
# fc2: 32*1 weights + 1 bias = 33
# Total: 385

# Forward pass
x_demo = torch.randn(4, 10)   # batch of 4 examples, 10 features each
out = net(x_demo)
print(f'Output shape: {out.shape}')  # (4, 1)

### 1d. The Standard Training Loop — Line by Line

Every PyTorch training loop follows the same 6-step pattern. This is worth
memorising verbatim for interviews.

In [ ]:
# Minimal training loop — annotated line by line
net   = TinyNet(10, 1)
opt   = torch.optim.Adam(net.parameters(), lr=1e-3)
crit  = nn.MSELoss()

x_fake = torch.randn(64, 10)   # 64 training examples, 10 features
y_fake = torch.randn(64, 1)    # 64 targets

for epoch in range(5):
    # 1. Set model to training mode (enables dropout, batch norm training behaviour)
    net.train()

    # 2. Zero the gradients from the previous iteration.
    #    PyTorch ACCUMULATES gradients — if you skip this, gradients pile up
    #    across iterations and updates become meaningless.
    opt.zero_grad()

    # 3. Forward pass: compute predictions.
    preds = net(x_fake)

    # 4. Compute loss (scalar).
    loss = crit(preds, y_fake)

    # 5. Backward pass: compute d(loss)/d(param) for every parameter.
    #    Gradients are stored in param.grad.
    loss.backward()

    # 6. Optimizer step: update param = param - lr * param.grad.
    opt.step()

    print(f'Epoch {epoch+1}  loss: {loss.item():.4f}')

---
## Section 2 — Why TCN over LSTM?

### The Mechanism Behind Each Property

**LSTM gates (brief review):**
At each timestep t, an LSTM computes:
```
i_t = sigmoid(W_i * [h_{t-1}, x_t] + b_i)  # input gate: how much new info to add
f_t = sigmoid(W_f * [h_{t-1}, x_t] + b_f)  # forget gate: how much old info to drop
o_t = sigmoid(W_o * [h_{t-1}, x_t] + b_o)  # output gate: how much to expose
g_t = tanh(W_g * [h_{t-1}, x_t] + b_g)      # candidate cell update
C_t = f_t * C_{t-1} + i_t * g_t             # update cell state
h_t = o_t * tanh(C_t)                        # output hidden state
```
Each timestep depends on the previous hidden state — **sequential** by design.
You cannot compute h_3 until you have h_2, which requires h_1, and so on.

**TCN (causal Conv1d):**
```
output[t] = conv_weights dot input[t - (k-1)*d : t : d]
```
All output timesteps can be computed **simultaneously** — no sequential dependency.
This is why TCN trains faster on GPU.

In [ ]:
comparison = pd.DataFrame([
    ('Parallelism', 'Sequential — h_t needs h_{t-1}',
     'Fully parallel — all t computed at once',
     'TCN trains faster on GPU; same speed on CPU'),
    ('Gradient path', 'Through time: multiply T weight matrices\n=> vanishing/exploding risk',
     'Residual shortcuts: d(loss)/d(x) = dF/dx + 1\n=> gradient >= 1 at every layer',
     'TCN is more stable to train'),
    ('Receptive field', 'Implicit via hidden state size\n(hard to control)',
     'Explicit: 1+(k-1)*2*sum(dilations)\n(you design it)',
     'Easier to reason about what history is used'),
    ('Parameters', 'O(4 * hidden^2) for 4 gate matrices',
     'O(channels * kernel_size) per block',
     'Comparable; often TCN is lighter'),
    ('Fixed-window input', 'Over-engineered (gates are for variable length)',
     'Natural fit — RF is tuned to window length',
     'Our use case: always 12 months -> TCN wins'),
    ('When LSTM wins', 'Truly variable-length history, NLP seq2seq',
     'Fixed-window forecasting, parallel series',
     'Know both — answer depends on task'),
], columns=['Property', 'LSTM', 'TCN', 'Implication'])

with pd.option_context('display.max_colwidth', 60, 'display.width', 200):
    print(comparison.to_string(index=False))

---
## Section 3 — Causal & Dilated Convolutions

### 3a. What `nn.Conv1d` does

`Conv1d(in_channels, out_channels, kernel_size)` slides a learnable kernel of
size `kernel_size` along the time axis:

```
output[t] = sum_k( weight[k] * input[t + k] )  for k in 0..kernel_size-1
```

**Problem:** with default padding, output[t] uses `input[t], input[t+1], ...,
input[t + kernel_size - 1]` — it looks **forward** into the future. For
time-series forecasting this is a data leak.

### 3b. Making it Causal

Add left padding only: pad `(kernel_size - 1) * dilation` zeros to the LEFT,
then trim the same amount from the RIGHT (that is `Chomp1d`).

Result: output[t] uses only `input[t - (k-1)*d], ..., input[t]`. ✓

### 3c. Dilation

Dilation skips positions:
```
dilation=1: output[t] uses input[t-2], input[t-1], input[t]         (kernel=3, adjacent)
dilation=2: output[t] uses input[t-4], input[t-2], input[t]         (skips every other)
dilation=4: output[t] uses input[t-8], input[t-4], input[t]         (skips 3)
```

Stacking dilation=1,2,4,8 gives an exponentially growing receptive field
without adding more parameters.

In [ ]:
# Visualise receptive fields for the 4 TemporalBlocks in CrimeTCN.
fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Receptive Field per TemporalBlock (kernel_size=3)', fontsize=14, fontweight='bold')

T = 12   # sequence length
kernel_size = 3
cumulative_rf = 0

for i, dilation in enumerate([1, 2, 4, 8]):
    ax = axes[i]
    ax.set_xlim(-0.5, T - 0.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_yticks([])
    ax.set_ylabel(f'Dilation={dilation}', rotation=0, ha='right', labelpad=60)
    ax.set_facecolor('#f0f4f8')

    # All positions in grey
    for t in range(T):
        ax.add_patch(plt.Rectangle((t - 0.4, 0.1), 0.8, 0.8,
                                    color='#cccccc', zorder=1))
        ax.text(t, 0.5, str(t + 1), ha='center', va='center',
                fontsize=8, color='#555555')

    # Highlight positions that feed into the final output timestep (T-1)
    # For a causal conv with this dilation, output[T-1] uses:
    # input[T-1 - (k-1)*d], input[T-1 - (k-2)*d], ..., input[T-1]
    rf_positions = [
        (T - 1) - (kernel_size - 1 - j) * dilation
        for j in range(kernel_size)
    ]
    rf_positions = [p for p in rf_positions if 0 <= p < T]

    # Block RF = 2 * (k-1) * dilation (two conv layers per block)
    block_start = max(0, (T - 1) - 2 * (kernel_size - 1) * dilation)
    for t in range(block_start, T):
        ax.add_patch(plt.Rectangle((t - 0.4, 0.1), 0.8, 0.8,
                                    color='#90CAF9', zorder=2))
        ax.text(t, 0.5, str(t + 1), ha='center', va='center',
                fontsize=8, color='#0D47A1')

    for t in rf_positions:
        ax.add_patch(plt.Rectangle((t - 0.4, 0.1), 0.8, 0.8,
                                    color='#1565C0', zorder=3))
        ax.text(t, 0.5, str(t + 1), ha='center', va='center',
                fontsize=8, color='white', fontweight='bold')

    rf = 1 + 2 * (kernel_size - 1) * dilation
    ax.set_title(f'Block {i+1}: per-block RF ≈ {rf} months'
                 f' | dark blue = direct connections to output[12]',
                 fontsize=9, loc='left', pad=2)

axes[-1].set_xlabel('Timestep (month in 12-month window)', fontsize=10)
plt.tight_layout()
plt.show()

total_rf = 1 + (kernel_size - 1) * 2 * (1 + 2 + 4 + 8)
print(f'\nTotal stacked RF = 1 + {kernel_size-1} * 2 * (1+2+4+8) = {total_rf} months')
print(f'Our window is only 12 months, so ALL input positions contribute to the output.')

---
## Section 4 — Residual Connections

### Why Deep Networks Without Residuals Fail

Consider stacking 8 layers: `output = f8(f7(f6(f5(f4(f3(f2(f1(x))))))))`.

The gradient of the loss wrt the first layer's parameters is:
```
d(loss)/d(W1) = d(loss)/d(f8) * d(f8)/d(f7) * ... * d(f2)/d(f1) * d(f1)/d(W1)
```

This is a product of 8 Jacobian matrices. If each has values < 1 (common
with sigmoid/tanh activations), the product vanishes exponentially. Gradients
near zero mean the first layers never learn — **vanishing gradient problem**.

### The Residual Fix

```
output = F(x) + x
```

Gradient via chain rule:
```
d(output)/d(x) = dF/dx + 1
```

The `+1` from the identity shortcut guarantees the gradient is at least 1 at
every block. The first layers always receive a useful gradient signal.

ReLU after the addition keeps values non-negative without saturating.

In [ ]:
# Demonstrate gradient flow with vs without residual connections.
torch.manual_seed(0)

# Build two 8-layer networks: one plain (no residual), one with residuals.
class PlainLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(16, 16)
    def forward(self, x):
        return torch.tanh(self.fc(x))   # tanh saturates: gradient < 1

class ResidualLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(16, 16)
    def forward(self, x):
        return torch.relu(self.fc(x)) + x  # identity shortcut

class PlainNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([PlainLayer() for _ in range(8)])
        self.out = nn.Linear(16, 1)
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.out(x)

class ResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([ResidualLayer() for _ in range(8)])
        self.out = nn.Linear(16, 1)
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.out(x)

x   = torch.randn(32, 16)
y   = torch.randn(32, 1)
crit = nn.MSELoss()

for name, net in [('PlainNet (tanh, no residual)', PlainNet()),
                  ('ResNet  (relu + residual)',   ResNet())]:
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    opt.zero_grad()
    loss = crit(net(x), y)
    loss.backward()

    # Inspect gradient magnitude in the FIRST layer (hardest to reach).
    first_layer_grad = net.layers[0].fc.weight.grad.abs().mean().item()
    last_layer_grad  = net.layers[7].fc.weight.grad.abs().mean().item()
    print(f'{name}')
    print(f'  First layer grad magnitude: {first_layer_grad:.6f}')
    print(f'  Last  layer grad magnitude: {last_layer_grad:.6f}')
    print(f'  Ratio (first/last):         {first_layer_grad/max(last_layer_grad,1e-9):.4f}')
    print()

---
## Section 5 — Hyperparameter Tuning Guide

For each hyperparameter: what it controls, how to tell if it is wrong, and what to do.

In [ ]:
guide = pd.DataFrame([
    ('kernel_size',
     'How many adjacent (dilated) timesteps each conv sees per layer',
     '3 is standard; 5 gives broader local patterns but more parameters',
     'Underfits locally (model misses short bursts) -> try 5'),
    ('num_channels (width)',
     'Number of feature maps in each TemporalBlock',
     '64 per layer; go wider (128) if model underfits',
     'val_loss >> train_loss = underfit -> wider; val_loss << = overfit -> narrower'),
    ('num_levels (depth)',
     'Number of stacked TemporalBlocks',
     '4 blocks; RF=61 months. For longer windows, add blocks',
     'RF should be >= your seq_len. RF = 1+(k-1)*2*sum(dilations)'),
    ('dropout',
     'Fraction of activations randomly zeroed during training',
     '0.2 is a safe default; range 0.1-0.4',
     'Overfitting (val rises, train falls) -> increase. Underfitting -> lower or 0'),
    ('learning_rate',
     'Step size for each gradient update',
     '1e-3 with Adam is a good starting point',
     'Loss explodes -> lower (1e-4). Loss plateaus immediately -> raise (3e-3)'),
    ('batch_size',
     'Number of sequences per gradient update',
     '32 is standard; 64 if GPU memory allows',
     'Smaller batches = noisier gradients but may escape local minima'),
    ('EPOCHS',
     'Number of full passes through the training data',
     '50 for this dataset; monitor val_loss for plateau',
     'Stop when val_loss has not improved for 5-10 epochs (early stopping)'),
    ('seq_len',
     'Length of the input window',
     '12 months (one year of history)',
     'Extend to 24 if you suspect 2-year seasonal cycles'),
], columns=['Hyperparameter', 'Controls', 'Default', 'Tuning Signal'])

with pd.option_context('display.max_colwidth', 55, 'display.width', 300):
    print(guide.to_string(index=False))

---
## Section 6 — Loss Functions

The choice of loss function changes what the model optimises.

| Loss | Formula | Sensitivity to outliers | Gradient behaviour |
|------|---------|------------------------|--------------------|
| MSE (L2) | mean( (y - ŷ)² ) | High — outliers are squared | Smooth; goes to 0 as error → 0 |
| MAE (L1) | mean( \|y - ŷ\| ) | Low — outliers treated equally | Constant (±1); slow near minimum |
| Huber | L2 if \|e\| ≤ δ, else δ(\|e\|-δ/2) | Medium | L2-smooth near 0, L1 far from 0 |

**Why we train with MSE but report MAE:**
- MSE has a smooth gradient → faster, more stable optimisation.
- MAE is more interpretable ("the model is off by X crimes on average").
- Huber is worth trying if a few high-crime areas dominate the MSE signal.

In [ ]:
errors = np.linspace(-50, 50, 300)

mse_loss   = errors ** 2
mae_loss   = np.abs(errors)
delta      = 10.0
huber_loss = np.where(
    np.abs(errors) <= delta,
    0.5 * errors ** 2,
    delta * (np.abs(errors) - 0.5 * delta)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(errors, mse_loss,   label='MSE (L2)',  linewidth=2)
axes[0].plot(errors, mae_loss,   label='MAE (L1)',  linewidth=2, linestyle='--')
axes[0].plot(errors, huber_loss, label=f'Huber (delta={delta})', linewidth=2, linestyle=':')
axes[0].set_ylim(0, 400)
axes[0].set_title('Loss vs Prediction Error')
axes[0].set_xlabel('Error (predicted - actual)')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].axvline(0, color='grey', lw=0.8, linestyle='--')

# Gradient of each loss
mse_grad   = 2 * errors
mae_grad   = np.sign(errors)
huber_grad = np.where(np.abs(errors) <= delta, errors, delta * np.sign(errors))

axes[1].plot(errors, mse_grad,   label='d(MSE)/d(error)',   linewidth=2)
axes[1].plot(errors, mae_grad,   label='d(MAE)/d(error)',   linewidth=2, linestyle='--')
axes[1].plot(errors, huber_grad, label='d(Huber)/d(error)', linewidth=2, linestyle=':')
axes[1].set_ylim(-60, 60)
axes[1].set_title('Gradient vs Prediction Error')
axes[1].set_xlabel('Error')
axes[1].set_ylabel('Gradient')
axes[1].legend()
axes[1].axhline(0, color='grey', lw=0.8, linestyle='--')
axes[1].axvline(0, color='grey', lw=0.8, linestyle='--')

plt.tight_layout()
plt.show()

---
## Section 7 — Full Pipeline Flowchart

This matplotlib figure shows the entire ML pipeline for the CityLiving Sim crime
prediction system, from raw CSV to the app-facing `predict()` output contract.

In [ ]:
fig = plt.figure(figsize=(24, 18))
ax  = fig.add_subplot(111)
ax.set_xlim(0, 24)
ax.set_ylim(0, 18)
ax.axis('off')
fig.patch.set_facecolor('#F5F7FA')
ax.set_facecolor('#F5F7FA')

# ── Colours ──────────────────────────────────────────────────────────────────
BLUE   = '#1565C0'   # data pipeline
GREEN  = '#2E7D32'   # TCN architecture
TEAL   = '#006064'   # TemporalBlock detail
ORANGE = '#E65100'   # training process
PURPLE = '#6A1B9A'   # residual
RED    = '#B71C1C'   # output
GRAY   = '#455A64'   # section headers

def box(ax, x, y, w, h, text, color, tcolor='white', fs=8.5, alpha=1.0, bold=False):
    """Draw a rounded box with centred text."""
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle='round,pad=0.12',
        facecolor=color, edgecolor='white',
        linewidth=1.8, alpha=alpha, zorder=2
    )
    ax.add_patch(patch)
    ax.text(
        x + w / 2, y + h / 2, text,
        ha='center', va='center',
        fontsize=fs, color=tcolor,
        fontweight='bold' if bold else 'normal',
        multialignment='center', zorder=3
    )

def arrow(ax, x1, y1, x2, y2, color='#455A64'):
    """Draw a simple arrow between two points."""
    ax.annotate(
        '', xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle='->', color=color, lw=1.8),
        zorder=4
    )

# ── Column 1: Data Pipeline (x=0.3 .. 4.3) ───────────────────────────────────
ax.text(2.3, 17.5, 'DATA PIPELINE', ha='center', fontsize=10,
        color=BLUE, fontweight='bold')

box(ax, 0.3, 16.2, 4.0, 0.9,
    'crime_aggregations.csv\n(community_id, year, month,\nprimary_type, crime_count)',
    BLUE, fs=7.5)
arrowow = arrow(ax, 2.3, 16.2, 2.3, 15.8)
arrowow = arrow(ax, 2.3, 15.8, 2.3, 15.25)
box(ax, 0.3, 14.35, 4.0, 0.85,
    'groupby(area, year, month)\nsum crime_count across types\n→ total monthly crimes per area',
    BLUE, fs=7.5)
arrow(ax, 2.3, 14.35, 2.3, 13.85)
box(ax, 0.3, 12.95, 4.0, 0.85,
    'Feature Engineering\nlag_1/2/3/6/12  rolling_3/6\nmonth_sin/cos  community_id',
    BLUE, fs=7.5)
arrow(ax, 2.3, 12.95, 2.3, 12.45)
box(ax, 0.3, 11.55, 4.0, 0.85,
    'Temporal Split\ntrain ≤2022 / val 2023 / test 2024',
    BLUE, fs=7.5)
arrow(ax, 2.3, 11.55, 2.3, 11.05)
box(ax, 0.3, 10.15, 4.0, 0.85,
    'StandardScaler\nfit on train only\ntransform val + test with same stats',
    BLUE, fs=7.5)
arrow(ax, 2.3, 10.15, 2.3, 9.65)
box(ax, 0.3, 8.75, 4.0, 0.85,
    'CrimeDataset\nsliding windows  seq_len=12\n→ (12, 10) input tensors',
    BLUE, fs=7.5)
arrow(ax, 2.3, 8.75, 2.3, 8.25)
box(ax, 0.3, 7.35, 4.0, 0.85,
    'DataLoader\nbatch_size=32  shuffle=True (train)\nshuffle=False (val / test)',
    BLUE, fs=7.5)

# ── Column 2: TCN Architecture (x=5.3 .. 9.3) ─────────────────────────────────
ax.text(7.3, 17.5, 'TCN ARCHITECTURE', ha='center', fontsize=10,
        color=GREEN, fontweight='bold')

box(ax, 5.3, 16.2, 4.0, 0.85,
    'Input\n(batch=32, seq_len=12, n_features=10)',
    GREEN, fs=7.5)
arrow(ax, 7.3, 16.2, 7.3, 15.7)
box(ax, 5.3, 14.8, 4.0, 0.85,
    'transpose(1, 2)\n(batch=32, n_features=10, seq_len=12)\nConv1d needs channels-first',
    GREEN, fs=7.5)
arrow(ax, 7.3, 14.8, 7.3, 14.3)
box(ax, 5.3, 13.4, 4.0, 0.85,
    'TemporalBlock  dilation=1\n→ (32, 64, 12)', GREEN, fs=7.5)
arrow(ax, 7.3, 13.4, 7.3, 12.9)
box(ax, 5.3, 12.0, 4.0, 0.85,
    'TemporalBlock  dilation=2\n→ (32, 64, 12)', GREEN, fs=7.5)
arrow(ax, 7.3, 12.0, 7.3, 11.5)
box(ax, 5.3, 10.6, 4.0, 0.85,
    'TemporalBlock  dilation=4\n→ (32, 64, 12)', GREEN, fs=7.5)
arrow(ax, 7.3, 10.6, 7.3, 10.1)
box(ax, 5.3, 9.2, 4.0, 0.85,
    'TemporalBlock  dilation=8\n→ (32, 64, 12)', GREEN, fs=7.5)
arrow(ax, 7.3, 9.2, 7.3, 8.7)
box(ax, 5.3, 7.8, 4.0, 0.85,
    'Last timestep  [:, :, -1]\n→ (32, 64)', GREEN, fs=7.5)
arrow(ax, 7.3, 7.8, 7.3, 7.3)
box(ax, 5.3, 6.4, 4.0, 0.85,
    'Linear(64 → 1)\n→ (32, 1)  predicted count', GREEN, fs=7.5)

# Arrow from DataLoader to TCN Input
ax.annotate('', xy=(5.3, 16.62), xytext=(4.3, 7.77),
            arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5,
                           connectionstyle='arc3,rad=-0.3'), zorder=4)

# ── Column 3: TemporalBlock Detail (x=10.5 .. 15.5) ──────────────────────────
ax.text(13.0, 17.5, 'TEMPORAL BLOCK (ONE LAYER)', ha='center',
        fontsize=10, color=TEAL, fontweight='bold')

box(ax, 10.5, 16.2, 5.0, 0.85,
    'Input (batch, in_channels, seq_len)', TEAL, fs=7.5)
arrow(ax, 13.0, 16.2, 13.0, 15.7)
box(ax, 10.5, 14.8, 5.0, 0.85,
    'Conv1d(dilation=d)\npadding=(kernel_size-1)*dilation\n→ keeps seq_len', TEAL, fs=7.5)
arrow(ax, 13.0, 14.8, 13.0, 14.3)
box(ax, 10.5, 13.4, 5.0, 0.85,
    'Chomp1d\nslice off right-side padding\n→ causal: output[t] uses only input[≤t]',
    TEAL, fs=7.5)
arrow(ax, 13.0, 13.4, 13.0, 12.9)
box(ax, 10.5, 12.0, 5.0, 0.85,
    'ReLU → Dropout', TEAL, fs=7.5)
arrow(ax, 13.0, 12.0, 13.0, 11.5)
box(ax, 10.5, 10.6, 5.0, 0.85,
    'Conv1d + Chomp1d + ReLU + Dropout\n(second conv, same dilation)', TEAL, fs=7.5)
arrow(ax, 13.0, 10.6, 13.0, 10.1)

# Residual path
box(ax, 10.5, 7.8, 5.0, 0.85,
    'Residual shortcut\n1×1 Conv if in_channels ≠ out_channels\nelse identity (free)', PURPLE, fs=7.5)

# Curved arrow: Input -> Residual shortcut
ax.annotate('', xy=(10.5, 8.22), xytext=(10.5, 16.62),
            arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=2.0,
                           connectionstyle='arc3,rad=0.4'), zorder=4)
ax.text(9.4, 12.4, 'skip\nconnection', ha='center', va='center',
        fontsize=8, color='#7B1FA2', style='italic')

# Main path converges
box(ax, 10.5, 9.2, 5.0, 0.85,
    'ADD (main + residual) → ReLU\nd(output)/d(x) = dF/dx + 1  ← no vanishing', PURPLE, fs=7.5)
arrow(ax, 13.0, 10.6, 13.0, 10.05)
arrow(ax, 13.0, 9.2, 13.0, 8.65)
arrow(ax, 13.0, 7.8, 13.0, 7.3)
box(ax, 10.5, 6.4, 5.0, 0.85,
    'Output (batch, out_channels, seq_len)', TEAL, fs=7.5)

# ── Column 4: Training + Output (x=16.5 .. 20.5) ─────────────────────────────
ax.text(18.5, 17.5, 'TRAINING & OUTPUT', ha='center', fontsize=10,
        color=ORANGE, fontweight='bold')

box(ax, 16.5, 16.2, 4.0, 0.85,
    'MSE Loss\n(predicted − actual)²\nSensitive to high-crime outliers', ORANGE, fs=7.5)
arrow(ax, 18.5, 16.2, 18.5, 15.7)
box(ax, 16.5, 14.8, 4.0, 0.85,
    'loss.backward()\nAutograd: computes ∂loss/∂param\nfor every weight in the network',
    ORANGE, fs=7.5)
arrow(ax, 18.5, 14.8, 18.5, 14.3)
box(ax, 16.5, 13.4, 4.0, 0.85,
    'clip_grad_norm_(max=1.0)\nScale gradient vector down if norm > 1\nPrevents destructive updates',
    ORANGE, fs=7.5)
arrow(ax, 18.5, 13.4, 18.5, 12.9)
box(ax, 16.5, 12.0, 4.0, 0.85,
    'optimizer.step()  (Adam)\nparam -= lr × grad\nAdaptive lr per parameter', ORANGE, fs=7.5)

# Output contract
ax.text(18.5, 11.2, '▼  OUTPUT CONTRACT  ▼', ha='center', fontsize=9,
        color=RED, fontweight='bold')
box(ax, 16.5, 9.6, 4.0, 1.3,
    'predict(community_area, year, month)\n─────────────────────────\npredicted_count: float\ntrend_direction: up | down | stable\nconfidence_score: 0.0 – 1.0',
    RED, fs=7.5)
arrow(ax, 18.5, 9.6, 18.5, 9.0)
box(ax, 16.5, 7.9, 4.0, 0.95,
    'queryMonthlyCrime.ts\n(Next.js API layer)\nReplaces hard-coded threshold rule', RED, fs=7.5)

# Arrow from TCN output to Training Loss
ax.annotate('', xy=(16.5, 16.62), xytext=(9.3, 6.82),
            arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5,
                           connectionstyle='arc3,rad=-0.25'), zorder=4)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color=BLUE,   label='Data Pipeline'),
    mpatches.Patch(color=GREEN,  label='TCN Architecture'),
    mpatches.Patch(color=TEAL,   label='TemporalBlock Internals'),
    mpatches.Patch(color=PURPLE, label='Residual Connection'),
    mpatches.Patch(color=ORANGE, label='Training Loop'),
    mpatches.Patch(color=RED,    label='Output / App Integration'),
]
ax.legend(handles=legend_patches, loc='lower left',
          fontsize=9, ncol=3, framealpha=0.85,
          bbox_to_anchor=(0.0, 0.0))

# ── Title ─────────────────────────────────────────────────────────────────────
ax.text(12.0, 18.0,
        'CityLiving Sim — TCN Crime Prediction Pipeline',
        ha='center', va='center', fontsize=16, fontweight='bold', color='#212121')

plt.tight_layout()
plt.savefig('tcn_pipeline_flowchart.png', dpi=150, bbox_inches='tight',
            facecolor='#F5F7FA')
plt.show()
print('Saved: tcn_pipeline_flowchart.png')

---
## Interview Cheat Sheet

**Q: What is a TCN?**  
A stack of TemporalBlocks, each with two dilated causal 1D convolutions and a residual shortcut. Exponentially growing dilation gives a large receptive field efficiently.

**Q: What makes a convolution causal?**  
Left-only padding + Chomp1d on the right. `output[t]` only sees `input[≤t]`.

**Q: Why weight norm instead of batch norm?**  
Batch norm requires stable batch statistics. At inference (batch_size=1), the estimates are meaningless. Weight norm reparameterizes weights without any batch dependency.

**Q: What is the receptive field of this model?**  
`1 + (kernel_size − 1) × 2 × sum(dilations) = 1 + 2×2×15 = 61 months.`

**Q: Why take only the last timestep before the linear layer?**  
Conv1d outputs a prediction at every timestep. We want the prediction for `t+1`
(next month), which is the output at position `t = seq_len − 1` (the final position
in the causal window).

**Q: How does autograd work?**  
PyTorch records every operation on tensors with `requires_grad=True` into a computation graph. `loss.backward()` traverses this graph in reverse using the chain rule to compute `∂loss/∂param` for every parameter.

**Q: Why shuffle training but not val/test?**  
Shuffling breaks autocorrelation between consecutive batches, which helps Adam's gradient estimates. Val/test must be deterministic for reproducible metrics.

**Q: What happens if you forget `optimizer.zero_grad()`?**  
Gradients accumulate across iterations. Each `backward()` adds to the previous gradient rather than replacing it. Weights update based on the sum of all past gradients — training diverges.